# Titanic Survival Prediction — Model Training & Comparison

This notebook trains and compares multiple classification models to predict
Titanic passenger survival, using the preprocessed data from `src/preprocessing.py`.

**Goal:** train several models, compare their performance using cross-validation,
and identify the strongest candidate(s) for further tuning.

**Baseline to beat:** 61.6% accuracy (always predicting "did not survive").

In [ ]:
import sys
sys.path.append('../src')
from preprocessing import full_pipeline

X_train, X_val, y_train, y_val = full_pipeline('../data/train.csv')
print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)

(712, 14) (179, 14) (712,) (179,)


In [6]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
predictions = model.predict(X_val)

In [9]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_val, predictions)
print(accuracy)

0.8156424581005587


In [12]:
import pandas as pd

coefficients = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient', ascending=False)

print(coefficients)

              Feature  Coefficient
8              Master     2.408691
3         Sex_encoded     2.158674
10                Mrs     1.515419
13           HasCabin     0.577723
9                Miss     0.435669
6                   C     0.416222
7                   Q     0.263078
11               Rare     0.187106
12               Fare     0.003521
4          Age_filled    -0.017176
5   Age_Group_encoded    -0.161133
2             isAlone    -0.269028
1          FamilySize    -0.466629
0              Pclass    -0.656286


In [22]:
lr_cv_scores = cross_val_score(model, X_train, y_train, cv=5)
print(lr_cv_scores)
print(lr_cv_scores.mean())

[0.84615385 0.84615385 0.83098592 0.78873239 0.83098592]
0.8286023835319609


# Logistic Regression

**What it is:** Despite the name, Logistic Regression is a *classification* model (predicts categories), not a regression model that predicts continuous numbers.

**How it works:**
1. Each feature (Pclass, Sex_encoded, Fare, etc.) is multiplied by a learned weight, and all these weighted features are summed together (plus an intercept) — producing a "raw score" that can be any number, positive or negative.
2. That raw score is passed through the **sigmoid function**, an S-shaped curve that squishes any input into a probability between 0 and 1:
   - Large positive raw score → probability close to 1 (likely survived)
   - Large negative raw score → probability close to 0 (likely did not survive)
   - Raw score of 0 → probability of exactly 0.5 (model is unsure)
3. If the resulting probability is above 0.5, the model predicts "survived" (1); otherwise, "did not survive" (0)

**Why it's a good baseline model:**
- Simple and fast to train
- Interpretable — the learned weights show which features push predictions toward survival vs. death, and by how much
- Gives a solid reference point to compare more complex models against later

**Training results:**
- `LogisticRegression(max_iter=1000)` — increased `max_iter` from the default 100 to resolve a convergence warning, likely caused by features being on very different scales (e.g., Fare ranges ~0-500, Sex_encoded is only 0 or 1)
- **Validation accuracy: 81.6%** — a strong improvement over the 61.6% baseline (always predicting "did not survive"); the accuracy means the model correctly predicted survived/death for about 81.6% of the 179 validation passengers
- Note: this accuracy comes from a single train/validation split (`random_state=42`), which can be somewhat lucky or unlucky depending on which passengers happened to land in validation

**Cross-validated result:**
```python
lr_cv_scores = cross_val_score(model, X_train, y_train, cv=5)
```
Fold scores: `[84.6%, 84.6%, 83.1%, 78.9%, 83.1%]` → **average: 82.9%**

This is a more trustworthy estimate than the original single-split result (81.6%), since it averages performance across 5 different train/test combinations rather than relying on one particular split. In this case, the original single split happened to be slightly *unlucky* — the true average performance is a bit higher than that one number suggested.

**Possible future improvement:** apply feature scaling (e.g., StandardScaler) to put all features on a comparable range, which may improve convergence and potentially model performance for Logistic Regression specifically (tree-based models like Random Forest are generally unaffected by feature scale).

**Coefficients used by Logistic Regression**

1. Strongest positive coefficients (push towards survival)
   - `Master` : 2.41, being a young boy strongly increases predicted survival probability
   - `Sex_encoded` : 2.16, being female strongly increases predicted survival probability
   - `Mrs` : 1.52, being a married woman strongly increases predicted survival probability
   - `HasCabin` : 0.58, having a recorded cabin increases predicted survival probability
2. Strongest negative coefficients (push towards death)
   - `Pclass` : -0.66, as class number increases 1 → 2 → 3, survival probability drops
   - `FamilySize` : -0.47, limitation of linear coefficient, since in EDA we saw that it is a curve (small families did better, only very large families did worse)
   - `isAlone` : -0.27, being alone decreases survival probability
3. Near-zero coefficients
   - `Fare` : 0.0035, nearly zero despite Fare showing a 0.26 correlation with survival in EDA. Fare was mostly a proxy for class/wealth, and `Pclass` (which already has a strong -0.66 coefficient) captures that same signal more directly — once Pclass is accounted for, Fare has little independent information left to contribute
   - `Age_filled` : -0.017, nearly zero despite the "children first" pattern found in EDA. Most of the strong age-related signal (specifically, young boys) is already captured by the `Master` feature (2.41). The remaining age-survival relationship, once young boys are excluded, is not a clean, single-direction trend across the rest of the age range — Logistic Regression can only fit one consistent slope per feature, so a relationship that's strong in one narrow sub-range but weak/inconsistent elsewhere ends up averaging out to a coefficient close to zero
   - `Age_Group_encoded` : -0.16, similarly small — likely for the same reason as Age_filled, since it's a more coarse-grained version of the same information

In [15]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier()
knn_model.fit(X_train_scaled, y_train)
knn_predictions = knn_model.predict(X_val_scaled)

from sklearn.metrics import accuracy_score
knn_accuracy = accuracy_score(y_val, knn_predictions)
print(knn_accuracy)

0.8268156424581006


In [ ]:
from sklearn.model_selection import cross_val_score

for k in [1, 3, 5, 7, 9, 11, 15, 21]:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_train_scaled, y_train, cv=5)
    print(k, scores.mean())

1 0.7570176302570669
3 0.8089530188121736
5 0.8117797695262483
7 0.8061459667093469
9 0.821609376538954
11 0.8159952723333005
15 0.8089825667290457
21 0.7893233527036344


In [19]:
knn_model = KNeighborsClassifier(n_neighbors=9)
knn_model.fit(X_train_scaled, y_train)
knn_predictions = knn_model.predict(X_val_scaled)

knn_accuracy = accuracy_score(y_val, knn_predictions)
print(knn_accuracy)

0.8379888268156425


## K-Nearest Neighbors (KNN)

**What it is:** Unlike Logistic Regression, KNN doesn't learn a formula or weights at all. Instead, to predict a new passenger's survival, it looks at the **K most similar passengers** in the training set (based on how close their feature values are, measured via Euclidean distance) and predicts whatever the majority of those similar passengers actually experienced.

**Why feature scaling was necessary:** KNN measures "similarity" using distance between feature values. If features have very different ranges (e.g., `Fare` spans 0-500, while `Sex_encoded` only spans 0-1), the feature with the larger range will dominate the distance calculation, even if it isn't actually the most important feature. Scaling puts all features on a comparable relative scale so each one contributes fairly to the similarity measurement (`StandardScaler` was used — mean=0, std=1 per feature, not a fixed 0-1 range).

**Training setup:**
```python
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
```
- `fit_transform()` on `X_train` — calculates scaling parameters (mean, standard deviation) **using only the training data**, then applies them
- `transform()` on `X_val` — applies those *same, already-learned* parameters to the validation set, without recalculating anything from it
- This avoids data leakage: the validation set should represent unseen data, so nothing in the pipeline (including preprocessing steps like scaling) should "learn" from it

**Choosing K via cross-validation:** K is a hyperparameter, not something the model learns automatically — it must be chosen manually. Tested K values [1, 3, 5, 7, 9, 11, 15, 21] using 5-fold cross-validation on the training set — for each K, the training data is split into 5 chunks, and 5 separate models are trained/tested (each on a different chunk combination), then their accuracy scores are averaged for a more reliable estimate than a single split:

| K | CV Accuracy |
|---|---|
| 1 | 75.7% |
| 3 | 80.9% |
| 5 | 81.2% |
| 7 | 80.6% |
| **9** | **82.2%** ← best |
| 11 | 81.6% |
| 15 | 80.9% |
| 21 | 78.9% |

- K=1 performs worst — too sensitive to a single, possibly atypical neighbor (overfitting)
- Large K (21) also underperforms — predictions start reflecting the overall dataset average rather than local patterns (underfitting)
- K=9 gave the best average cross-validated performance, so it was chosen as the final K

**Final model: KNN with K=9**, trained on the full training set, evaluated on the untouched validation set.

**Result: 83.8% single-split validation accuracy, 82.2% cross-validated average**
- The single-split result (83.8%) is higher than the cross-validated average (82.2%) — a reminder that a single split can be somewhat lucky, and the cross-validated number is the more trustworthy estimate of true performance
- Comparing fairly (cross-validated numbers for both): Logistic Regression (82.9%) actually slightly **outperforms** KNN (82.2%) — the opposite conclusion from comparing single-split numbers alone. This is a good illustration of why consistent evaluation methodology matters when comparing models

**Updated model comparison (cross-validated):**

| Model | CV Accuracy |
|---|---|
| Baseline | 61.6% |
| Logistic Regression | 82.9% |
| KNN (K=9) | 82.2% |

In [20]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_val)

rf_accuracy = accuracy_score(y_val, rf_predictions)
print(rf_accuracy)

0.8156424581005587


In [21]:
from sklearn.model_selection import cross_val_score

rf_cv_scores = cross_val_score(rf_model, X_train, y_train, cv=5)
print(rf_cv_scores)
print(rf_cv_scores.mean())

[0.81118881 0.77622378 0.8028169  0.80985915 0.83802817]
0.8076233625529401


## Random Forest

**What it is:** Random Forest is an *ensemble* method — instead of one single model, it builds many individual **decision trees** (often hundreds), each trained on a random subset of data and features, and combines their predictions (typically by majority vote) into one final prediction.

**How a decision tree works:** a tree makes predictions by asking a series of threshold-based yes/no questions about features — e.g., "Is Sex_encoded = 1?", "Is Fare > 50?" — progressively splitting passengers into smaller, more homogeneous groups until it reaches a final prediction. The algorithm automatically searches for the best threshold to split on for each feature, rather than checking exact-match values.

**Why feature scaling was NOT necessary:** unlike KNN, a decision tree evaluates one feature at a time in isolation (e.g., "is Fare > 50?") — it never mathematically combines multiple features into a single calculation the way KNN's distance formula does. Since there's no cross-feature magnitude comparison happening, the raw scale of any given feature doesn't affect how well the tree can find useful thresholds. `X_train`/`X_val` (unscaled) were used directly.

**Why an ensemble of trees, rather than one tree:** a single decision tree can overfit — asking increasingly specific questions until it essentially memorizes the training data rather than learning generalizable patterns. Random Forest averages together many different trees (each seeing different random subsets of data/features), smoothing out any individual tree's overfitting tendencies.

**Training setup:**
```python
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(random_state=42)
```
- `random_state=42` — Random Forest involves internal randomness (random sampling of data/features per tree), so this ensures reproducible results across runs, same purpose as in `train_test_split`

**Result: 81.6% single-split validation accuracy**

**Cross-validated result:**
Fold scores: `[81.1%, 77.6%, 80.3%, 81.0%, 83.8%]` → **average: 80.8%**

The cross-validated average (80.8%) is slightly lower than the single-split result (81.6%) — another example of a single split being somewhat optimistic compared to the true average performance. There's also a noticeable spread across folds (77.6% to 83.8%, roughly a 6-point range), suggesting some sensitivity to which passengers land in training vs. testing.

**Updated model comparison (cross-validated):**

| Model | CV Accuracy |
|---|---|
| Baseline | 61.6% |
| Logistic Regression | 82.9% |
| KNN (K=9) | 82.2% |
| Random Forest (default settings) | 80.8% |

**Note:** Random Forest was trained using default settings (default number of trees, unrestricted tree depth, etc.). This is a strong candidate for hyperparameter tuning in Phase 7, since tree-based models often have more room for improvement through tuning number of trees, max depth, and other parameters, compared to simpler models like Logistic Regression.